In [12]:
import overpy
import pandas as pd

In [13]:
df_city = pd.read_csv('TI/worldcities.csv')

In [102]:
df_capitals = df_city[df_city['capital'] == "primary"].reset_index(drop=True)

In [104]:
df_capitals.drop(df_capitals.index[206], inplace=True)

In [92]:
import urllib, json
def bbox_city(name):
    link = "https://nominatim.openstreetmap.org/search/" + str(name).replace(" ","-") + "?format=geojson&polygon=1&polygon_geojson=1&limit=1"
    response = urllib.request.urlopen(link)
    data = json.loads(response.read())
    result = data['features'][0]['bbox']
    return result

In [106]:
df_capitals['bbox'] = df_capitals['city_ascii'].apply(bbox_city)

In [115]:
df_capitals.to_csv('capitals_bbox.csv')

In [112]:
def clean_bbox(result):
    myorder = [1, 0, 3, 2]
    result_clean = [result[i] for i in myorder]
    result_clean = str(result_clean)
    bbox_clean = result_clean.replace("[","")
    bbox_clean = bbox_clean.replace("]","")
    bbox_clean = str(bbox_clean)
    return bbox_clean

In [113]:
df_capitals['bbox'] = df_capitals['bbox'].apply(clean_bbox)

Trees

In [259]:
import time
def trees(bbox_clean):

#a = "\"\"\"[out:json][timeout:25];(node[\"natural\"=\"tree\"]("
#b = "\););out tags;>;\"\"\""
#bbox_list = list(map(float, list_raw))
    query = 'https://lz4.overpass-api.de/api/interpreter?data=[out:json];(node["natural"="tree"]('+bbox_clean+'););out body;>;out skel qt;'
    response = requests.get(query)
    print(response.status_code)
    return response.json()

In [469]:
import json
number = 203
name = df_capitals['city_ascii'][number]
data = trees(df_capitals['bbox'][number])
                                    
with open(name+'.json', 'w') as convert_file:
     convert_file.write(json.dumps(data))

200


In [ ]:
'[out:json][timeout:25];(node["natural"="tree"]('20.2145811, 135.8536855, 35.8984245, 154.205541'););out tags;>;'

node
  ["natural"="tree"]
  (20.2145811, 135.8536855, 35.8984245, 154.205541);
out;

[out:json];(node["natural"="tree"](20.2145811, 135.8536855, 35.8984245, 154.205541););out body;>;out skel qt;

In [174]:
df_capitals.to_csv("capitals_final.csv")

In [136]:
trees('20.2145811, 135.8536855, 35.8984245, 154.205541')

OverpassRuntimeError: runtime error: Query timed out in "print" at line 1 after 27 seconds.

In [120]:
pip install overpass

3847.37s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 25.8 MB/s eta 0:00:0000:01
Note: you may need to restart the kernel to use updated packages.


In [ ]:
https://lz4.overpass-api.de/api/interpreter?data=[out:json];(node["natural"="tree"](20.2145811,%20135.8536855,%2035.8984245,%20154.205541););out%20body;>;out%20skel%20qt;

In [138]:
import requests

In [9]:
####Zipping json
import pandas as pd
import json
df_capitals = pd.read_csv("capitals_final.csv")
path = "trees_capitals/"
name = number = 100
name = df_capitals['city_ascii'][number]

df = pd.read_json(path+name+'.json',orient='index')
trees = pd.DataFrame(df[0]['elements'])
trees = trees.sort_values(by=['lat','lon'], ascending=True)
sample = trees[0:999].reset_index(drop=True)
sample.head()

,type,id,lat,lon,tags
0,node,1211333992,18.452932,-72.348793,"{'natural': 'tree', 'source': 'Bing; survey'}"
1,node,1211125136,18.457998,-72.351077,"{'natural': 'tree', 'source': 'Bing'}"
2,node,640734182,18.508472,-72.508140,{'natural': 'tree'}
3,node,640720955,18.508578,-72.507749,{'natural': 'tree'}
4,node,640720929,18.508589,-72.507910,{'natural': 'tree'}


In [10]:
import folium
import pandas as pd

import folium.plugins as plugins
 
CITY_COORDINATES = (df_capitals['lat'][number], df_capitals['lng'][number])

# for speed purposes
MAX_RECORDS = 1000

token = "YOUR_MAPBOX_TOKEN"
tileurl = 'https://api.mapbox.com/v4/mapbox.satellite/{z}/{x}/{y}@2x.png?access_token=' + str(token)

# create empty map zoomed in on the city
map = folium.Map(location=CITY_COORDINATES, zoom_start=12,tiles=tileurl, attr='Mapbox')

# add a marker for every record in the filtered data, use a clustered view
for each in sample[0:MAX_RECORDS].iterrows():
    folium.Marker([each[1]['lat'],each[1]['lon']], tooltip=each[1]['lat']).add_to(map)
                                    
display(map)

In [2]:
#NEW METHOD FOR CALCULATING BBOX 
lon = 5.011857
lat = 52.349148

calc_1 = 4.965501-4.965418
calc_2 = 52.3233715-52.323340
calc_3 = 4.965501-4.965584
calc_4 = 52.3233715-52.323403

bbox_val1 = lon+calc_1
bbox_val2 = lat+calc_2

bbox_val3 = lon+calc_3
bbox_val4 = lat+calc_4

bbox = [bbox_val1,bbox_val2,bbox_val3
            ,bbox_val4]

In [3]:
#create outside geometry polygon

from shapely.geometry import box
#b = box(4.965418,52.323340,4.965584,52.323403)
b = box(bbox_val1,bbox_val2,bbox_val3
            ,bbox_val4)
points = list(b.exterior.coords)

print(b.wkt)

POLYGON ((5.011774 52.3491795, 5.011774 52.3491165, 5.01194 52.3491165, 5.01194 52.3491795, 5.011774 52.3491795))


In [4]:
import geopandas
json = geopandas.GeoSeries(b).to_json()

In [5]:
geo_file = geopandas.GeoSeries(b)
geo_file.crs = "epsg:4326"
geo_file = geo_file.to_crs("EPSG:3857")
geo_file.to_json()

'{"type": "FeatureCollection", "features": [{"id": "0", "type": "Feature", "properties": {}, "geometry": {"type": "Polygon", "coordinates": [[[557908.1296509678, 6863509.512273038], [557908.1296509678, 6863498.031304486], [557926.6086864396, 6863498.031304486], [557926.6086864396, 6863509.512273038], [557908.1296509678, 6863509.512273038]]]}, "bbox": [557908.1296509678, 6863498.031304486, 557926.6086864396, 6863509.512273038]}], "bbox": [557908.1296509678, 6863498.031304486, 557926.6086864396, 6863509.512273038]}'

In [ ]:
geo_file.to_file("testdata.geojson")

In [14]:
def create_bbox(row):
  lon = row['lon']
  lat = row['lat']
  
  calc_1 = 4.965501-4.965418
  calc_2 = 52.3233715-52.323340
  calc_3 = 4.965501-4.965584
  calc_4 = 52.3233715-52.323403

  bbox_val1 = lon+calc_1
  bbox_val2 = lat+calc_2
  bbox_val3 = lon+calc_3
  bbox_val4 = lat+calc_4

  bbox = [bbox_val1,bbox_val2,bbox_val3
            ,bbox_val4]
  b = box(bbox_val1,bbox_val2,bbox_val3
            ,bbox_val4)
  points = list(b.exterior.coords)
  result = geopandas.GeoSeries(b)
  return result


In [30]:
sample['polygon'] = sample.apply(create_bbox, axis=1)
geo_df = sample[['id','polygon']]
import geopandas
gdf = geopandas.GeoDataFrame(
    geo_df, geometry = "polygon")
gdf.crs = "epsg:4326"
gdf = gdf.to_crs("EPSG:3857")
#api_call_data = gdf.to_json()
gdfjson = gdf.to_json()

In [34]:
with open('testapi.json', 'w') as convert_file:
     convert_file.write(json.dumps(gdfjson))

In [98]:
from oauthlib.oauth2 import BackendApplicationClient
from requests_oauthlib import OAuth2Session


from sentinelhub import SHConfig

config = SHConfig()

# Your client credentials
client_id = '96acbb95-6c77-4d93-aaf3-ca218727e1b4'
client_secret = 'F@,ZbT*8PTZ(F}:7@.|,*;#K/K/Ev+#)uRLQ,^t2'

# Create a session
client = BackendApplicationClient(client_id=client_id)
oauth = OAuth2Session(client=client)

# Get token for the session
token = oauth.fetch_token(token_url='https://services.sentinel-hub.com/oauth/token',
                          client_secret=client_secret)

# All requests using this session will have an access token automatically added
resp = oauth.get("https://services.sentinel-hub.com/oauth/tokeninfo")
print(resp.content)

b'{"sub":"f71bef76-9296-41ab-90ac-459ffdc5f3ac","aud":"96acbb95-6c77-4d93-aaf3-ca218727e1b4","jti":"a01a5bcc-cf4e-4138-b586-2a723b9c814c","exp":1670328146,"name":"Jakub Supera","email":"jakub.supera@gmail.com","given_name":"Jakub","family_name":"Supera","sid":"1acca238-a75b-461e-9152-bcd0329059a2","did":1,"aid":"1f45fa8a-dd28-4373-bae1-3ea4c83e1c78","d":{"1":{"ra":{"rag":1},"t":12000}},"active":true}'


In [102]:
auth = token['access_token']


In [110]:
import requests

headers = {
    'Authorization': 'Bearer ' +auth ,
}

files = {
    'request': (None, '{\n    "input": {\n        "bounds": {\n            "properties": {\n                "crs": "http://www.opengis.net/def/crs/OGC/1.3/CRS84"\n            },\n            "geometry": {\n                "type": "Polygon",\n                "coordinates": [\n                         [\n        [4.965649, 52.335444],\n        [5.048904, 52.335444],\n        [5.048904, 52.367424],\n        [4.965649, 52.367424],\n        [4.965649, 52.335444]\n      ]\n                       \n                ]\n            }\n        },\n        "data": [\n            {\n                "type": "sentinel-2-l2a",\n                "dataFilter": {\n                    "timeRange": {\n                        "from": "2022-07-06T00:00:00Z",\n                        "to": "2022-07-07T00:00:00Z"\n                    }\n                },\n                "processing": {\n                    "harmonizeValues": "true"       \n                }\n            }\n        ]\n    },\n    "output": {\n        "width": 512,\n        "height": 512,\n        "responses": [\n            {\n                "identifier": "default",\n                "format": {\n                    "type": "image/tiff"\n                }\n            }\n        ]\n    }\n}'),
    'evalscript': (None, '//VERSION=3\nfunction setup() {\n  return{\n    input: [{\n      bands: ["B04", "B08"],\n      units: "REFLECTANCE"\n    }],\n    output: {\n      id: "default",\n      bands: 1,\n      sampleType: SampleType.FLOAT32\n    }\n  }\n}\n\nfunction evaluatePixel(sample) {\n  let ndvi = (sample.B08 - sample.B04) / (sample.B08 + sample.B04)\n  return [ ndvi ]\n}'),
}

response = requests.post('https://services.sentinel-hub.com/api/v1/process', headers=headers, files=files)

In [111]:
response.status_code

200

In [112]:
with open("response6.tif", "wb") as f:
    f.write(response.content)

In [4]:
import rasterio
from rasterio.plot import show
fp = r'response.tif'
img = rasterio.open(fp)
show(img)

ModuleNotFoundError: No module named 'rasterio'

In [17]:
!pip install python-gdal


ERROR: Could not find a version that satisfies the requirement python-gdal (from versions: none)
ERROR: No matching distribution found for python-gdal
